<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03c_visualizations_net_change.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing

In [69]:

import pandas as pd
import geopandas as gpd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import plotly.express as px


from google.colab import drive
drive.mount('/content/drive')

!wget https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
import functions as fx

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2026-04-12 20:42:20--  https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2015 (2.0K) [text/plain]
Saving to: ‘functions.py.3’

functions.py.3      100%[===================>]   1.97K  --.-KB/s    in 0s      

2026-04-12 20:42:21 (31.7 MB/s) - ‘functions.py.3’ saved [2015/2015]



##Reading in file

In [70]:
gdf_open_close = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/open_close_after_2016.geojson")
gdf_biz = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/biz_all_startdate.geojson")
#immediately clipping to 2025
gdf_biz = gdf_biz[(gdf_biz.year_open <=2025)]

In [71]:
gdf_biz.columns

Index(['uniqueid', 'business_account_number', 'location_id', 'ownership_name',
       'dba_name', 'business_start_date', 'business_end_date',
       'location_start_date', 'location_end_date', 'naics_code',
       'naics_code_description', 'lic_code', 'lic_code_description',
       'business_corridor', 'neighborhoods_analysis_boundaries',
       'administratively_closed_bool', 'status', 'year_open', 'year_closed',
       'lon', 'lat', 'geometry'],
      dtype='object')

##Reading in Census Tracts

In [72]:
sf_tracts = gpd.read_file('/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts.geojson')
sf_block_grp = gpd.read_file('/content/drive/MyDrive/C255_final_project/cleaned/sf_block_grp.geojson')
sf_poly = gpd.read_file('/content/drive/MyDrive/C255_final_project/cleaned/sf_poly.geojson')
sf_block_grp = sf_block_grp.rename(columns={'geoid': 'GEOID'})

##Joining census tracts

In [73]:
open_close_tracts_gdf = fx.group_points_by_poly_year(gdf_open_close, sf_tracts)

##Calculating new metrics


In [74]:
## net change from prior year

open_close_tracts_gdf['net_change'] = open_close_tracts_gdf['opened'] - open_close_tracts_gdf['closed']

# get 2016 baseline per tract
baseline = open_close_tracts_gdf[open_close_tracts_gdf['year'] == 2016][['GEOID', 'net_change']].rename(columns={'net_change': 'baseline_2016'})

# merge
open_close_tracts_gdf = open_close_tracts_gdf.merge(baseline, on='GEOID')



In [75]:
#net change compared to 2016
open_close_tracts_gdf['growth_pct_over_2016'] = (open_close_tracts_gdf['net_change'] / open_close_tracts_gdf['baseline_2016']) * 100

#total activity - openings plus closings
open_close_tracts_gdf['total_activity'] = (
    open_close_tracts_gdf['closed'] + open_close_tracts_gdf['opened']
)

In [76]:
##getting total number of businesses active in each year ()

gdf_biz['year_closed'] = gdf_biz['year_closed'].fillna(2025).astype(int)
gdf_biz['year_open'] = gdf_biz['year_open'].astype(int)

gdf_biz['active_years'] = gdf_biz.apply(
    lambda row: list(range(row['year_open'], row['year_closed'] + 1)), axis=1
)

biz_exploded = gdf_biz.explode('active_years').rename(columns={'active_years': 'year'})

biz_exploded = gpd.sjoin(biz_exploded, sf_tracts[['GEOID', 'geometry']], how='left', predicate='within')

biz_stock = biz_exploded.groupby(['GEOID', 'year']).size().reset_index(name='biz_stock')

open_close_tracts_gdf = open_close_tracts_gdf.merge(biz_stock, on=['GEOID', 'year'], how='left')

In [77]:
open_close_tracts_gdf['net_entry_rate'] = (open_close_tracts_gdf['net_change'] / open_close_tracts_gdf['biz_stock'])*100
open_close_tracts_gdf['gross_exit_rate'] = (open_close_tracts_gdf['opened'] / open_close_tracts_gdf['biz_stock'])*100

In [78]:
open_close_tracts_gdf

,GEOID,geometry,year,closed,opened,net_change,baseline_2016,growth_pct_over_2016,total_activity,prior_year_net,pct_change_prior_year,biz_stock,net_entry_rate,gross_exit_rate
0,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2016.0,26.0,87.0,61.0,61.0,100.000000,113.0,NaN,NaN,402,15.174129,21.641791
1,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2017.0,40.0,68.0,28.0,61.0,45.901639,108.0,61.0,-54.098361,444,6.306306,15.315315
2,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2018.0,61.0,85.0,24.0,61.0,39.344262,146.0,28.0,-14.285714,489,4.907975,17.382413
3,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2019.0,52.0,77.0,25.0,61.0,40.983607,129.0,24.0,4.166667,505,4.950495,15.247525
4,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",2020.0,59.0,24.0,-35.0,61.0,-57.377049,83.0,25.0,-240.000000,477,-7.337526,5.031447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2391,06075980900,"POLYGON ((-122.40459 37.74413, -122.40356 37.7...",2021.0,103.0,152.0,49.0,107.0,45.794393,255.0,63.0,-22.222222,1308,3.746177,11.620795
2392,06075980900,"POLYGON ((-122.40459 37.74413, -122.40356 37.7...",2022.0,110.0,115.0,5.0,107.0,4.672897,225.0,49.0,-89.795918,1320,0.378788,8.712121
2393,06075980900,"POLYGON ((-122.40459 37.74413, -122.40356 37.7...",2023.0,95.0,107.0,12.0,107.0,11.214953,202.0,5.0,140.000000,1317,0.911162,8.124525
2394,06075980900,"POLYGON ((-122.40459 37.74413, -122.40356 37.7...",2024.0,118.0,72.0,-46.0,107.0,-42.990654,190.0,12.0,-483.333333,1294,-3.554869,5.564142


#Choropleth maps

###getting epc tracts and adding is_epc boolean

In [83]:
epc_tracts = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/epc_tracts_sf.geojson")
epc_tracts = epc_tracts.rename(columns={'tract_geoid': 'GEOID'})
open_close_tracts_gdf['is_epc'] = open_close_tracts_gdf['GEOID'].isin(epc_tracts['GEOID'])

###Net entry rate map (net change for year / total businesses active that year)

In [91]:
# import plotly.graph_objects as go

# plot_gdf = open_close_tracts_gdf[open_close_tracts_gdf['year'] >= 2016].copy()


# vabs = open_close_tracts_gdf['net_entry_rate'].abs().quantile(.99)

# fig_over_time = px.choropleth_mapbox(
#     plot_gdf,
#     geojson=plot_gdf.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="net_entry_rate",
#     hover_name="GEOID",
#     hover_data={'is_epc': True, 'opened': True, 'closed': True, 'gross_exit_rate': True, 'biz_stock':True},
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="RdBu",
#     color_continuous_midpoint=0,
#     range_color=[-vabs, vabs],
#     height=600,
#     width=700
# )

# epc_outline = plot_gdf[plot_gdf['is_epc']]  # fixed: filter from plot_gdf
# fig_over_time.add_trace(go.Choroplethmapbox(
#     geojson=epc_outline.set_index("GEOID").__geo_interface__,
#     locations=epc_outline["GEOID"],
#     z=[1] * len(epc_outline),
#     colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
#     marker_line_color='black',
#     marker_line_width=3,
#     showscale=False,
#     hoverinfo='skip',
#     name='EPC Tracts'
# ))

# fig_over_time.show()


###Map of percent change in businesses compared to 2016 (net change each year / baseline net change in 2016)

In [90]:

# plot_gdf = open_close_tracts_gdf[open_close_tracts_gdf['year'] >= 2017].copy()

# vabs = plot_gdf['growth_pct_over_2016'].abs().quantile(0.95)

# fig_over_time = px.choropleth_mapbox(
#     plot_gdf,
#     geojson=plot_gdf.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="growth_pct_over_2016",
#     hover_name="GEOID",
#     hover_data={'is_epc': True, 'opened': True, 'closed': True, 'total_activity': True},
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="RdBu",
#     color_continuous_midpoint=0,
#     range_color=[-vabs, vabs],
#     height=600,
#     width=700
# )

# epc_outline = plot_gdf[plot_gdf['is_epc']]
# fig_over_time.add_trace(go.Choroplethmapbox(
#     geojson=epc_outline.set_index("GEOID").__geo_interface__,
#     locations=epc_outline["GEOID"],
#     z=[1] * len(epc_outline),
#     colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
#     marker_line_color='black',
#     marker_line_width=3,
#     showscale=False,
#     hoverinfo='skip',
#     name='EPC Tracts'
# ))

# fig_over_time.show()
